# SilentBridge — train real base_model.pt

Dataset: **ISL-CSLTR** (Kaggle `drblack00/isl-csltr-indian-sign-language-dataset`).

**Known issue with this Kaggle mirror**: it ships `Frames_Word_Level/` — a handful of
static jpgs per isolated word, NOT sentence-level video clips, even though the
dataset paper describes 700 sentence videos. This notebook checks what's actually
in the download and makes you confirm before it builds anything — it will not
silently guess a folder layout again.

**If you already hit `AttributeError: module 'mediapipe' has no attribute
'solutions'` or `ImportError: cannot import name 'runtime_version' from
'google.protobuf'`**: both are dependency-pin issues, fixed in cell 1
(mediapipe==0.10.14 + protobuf>=4.25.3). If you already imported mediapipe or
hit a broken path once in this session, **restart the runtime** (Runtime >
Restart session) so nothing stale is cached, then run all cells from the top.
Cell 2 will also auto-detect and fix a corrupted/nested clone from before this
fix landed.

Runtime: Runtime > Change runtime type > T4 GPU.

In [ ]:
# 1. Install deps not already in Colab.
# mediapipe pinned to 0.10.14 — newer 0.10.2x+ builds dropped the legacy
# `mp.solutions` API by default on Python 3.11/3.12 runtimes (AttributeError:
# module 'mediapipe' has no attribute 'solutions'), which is what
# extract_keypoints.py and the word-frames fallback both use.
# protobuf>=5.28 — the runtime_version submodule mediapipe 0.10.14's tasks
# import needs was only added in protobuf 5.26+, and Colab's own tensorflow
# already requires >=5.28 anyway, so pinning lower (as an earlier version of
# this notebook did) fights Colab's preinstalled stack instead of fixing it.
!pip install -q "mediapipe==0.10.14" "protobuf>=5.28.0,<6" opencv-python-headless pandas kagglehub

In [ ]:
# 2. Clone the repo — reuse its exact model code / training script / tokenizer.
# Anchored to a FIXED base dir (not the current cwd) so this is self-healing
# even if a previous run left cwd in a corrupted nested state — always ends
# up at exactly BASE_DIR/SilentBridge, never .../SilentBridge/SilentBridge.
import os

BASE_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_ROOT = os.path.join(BASE_DIR, "SilentBridge")

if not os.path.isdir(REPO_ROOT):
    os.chdir(BASE_DIR)
    !git clone https://github.com/BharathWaj-K-R/SilentBridge.git
elif not os.path.isfile(os.path.join(REPO_ROOT, "README.md")):
    # exists but looks corrupted/incomplete (e.g. old nested-clone state) — nuke and reclone
    !rm -rf {REPO_ROOT}
    os.chdir(BASE_DIR)
    !git clone https://github.com/BharathWaj-K-R/SilentBridge.git

os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)

In [ ]:
# 3. Get the dataset path — WITHOUT re-downloading if it's already mounted.
# On Kaggle Notebooks, adding this dataset as a notebook input mounts it
# read-only at /kaggle/input/<slug> for free — no download, no storage used.
# Re-running kagglehub.dataset_download() anyway is what was filling your
# working-storage quota every run. On Colab there's no such mount, so it
# falls back to the real download there.
import glob

_kaggle_mounts = glob.glob("/kaggle/input/isl-csltr-indian-sign-language-dataset*")
if _kaggle_mounts:
    dataset_path = _kaggle_mounts[0]
    print("using existing Kaggle input mount (no download):", dataset_path)
else:
    import kagglehub
    dataset_path = kagglehub.dataset_download("drblack00/isl-csltr-indian-sign-language-dataset")
    print("downloaded to:", dataset_path)

In [ ]:
# 3b. Diagnostic — run this any time storage looks off. Shows exactly what's
# using space in the working directory vs the (read-only, free) dataset mount.
os.chdir(REPO_ROOT)
print("=== du -sh of REPO_ROOT/data (this is what accumulates) ===")
!du -sh {REPO_ROOT}/data 2>/dev/null || echo "(no data/ dir yet)"
!du -sh {REPO_ROOT}/data/* 2>/dev/null
print("\n=== overall disk usage ===")
!df -h {REPO_ROOT} 2>/dev/null || df -h .

## 3c. Reset — wipe all working data, start clean

Optional, run only when you want a clean slate (e.g. switching DATA_MODE, or
cleaning up after a bad run). Deletes everything this notebook wrote —
`data/raw_videos`, `data/labels`, `data/processed`, and the trained
`base_model.pt`/`base_model.vocab.json` — but leaves the read-only dataset
mount (`/kaggle/input/...` or the kagglehub cache) untouched, so cell 3
won't need to re-download anything after this.

In [ ]:
import shutil

os.chdir(REPO_ROOT)
for path in [
    "data/raw_videos",
    "data/labels",
    "data/processed",
    "backend/app/models/weights/base_model.pt",
    "backend/app/models/weights/base_model.vocab.json",
]:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print("removed dir:", path)
    elif os.path.isfile(path):
        os.remove(path)
        print("removed file:", path)
print("clean. re-run from cell 5 (DATA_MODE) onward.")

In [ ]:
# 4. Inspect — shallow tree (won't get lost one branch deep like before),
# plus a whole-tree extension count and a search for anything sentence-level.
import os
from collections import Counter

print("=== shallow tree (depth <= 2) ===")
for root, dirs, files in os.walk(dataset_path):
    depth = root.replace(dataset_path, "").count(os.sep)
    if depth > 2:
        dirs[:] = []
        continue
    print("  " * depth + os.path.basename(root) + f"/  ({len(files)} files, {len(dirs)} subdirs)")

print("\n=== file extension counts (whole tree) ===")
ext_counts = Counter()
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        ext_counts[os.path.splitext(f)[1].lower()] += 1
for ext, n in ext_counts.most_common():
    print(f"{ext or '(no ext)'}: {n}")

print("\n=== folders with 'sentence' in the name ===")
found_sentence_dir = False
for root, dirs, files in os.walk(dataset_path):
    for d in dirs:
        if "sentence" in d.lower():
            print(os.path.join(root, d))
            found_sentence_dir = True
if not found_sentence_dir:
    print("(none found)")

print("\n=== ISL_CSLRT.txt (if present) ===")
for root, dirs, files in os.walk(dataset_path):
    if "ISL_CSLRT.txt" in files:
        p = os.path.join(root, "ISL_CSLRT.txt")
        print(p)
        with open(p, encoding="utf-8", errors="replace") as fh:
            for i, line in enumerate(fh):
                print(line.rstrip())
                if i > 30:
                    print("...")
                    break

## 5. REQUIRED — read cell 4's output, then fill this in yourself

Two possible modes, based on what you actually have:

- **`"video"`** — you found real `.mp4`/`.avi`/`.mov` sentence-level clips (a real
  video folder, not `Frames_Word_Level`). Best case — continuous-sentence training,
  matches the project's actual pitch. Set `VIDEO_ROOT` to that folder.
- **`"word_frames"`** — all you have is `Frames_Word_Level/<WORD>/*.jpg` (the case
  in this mirror as of when this notebook was written). Fallback: each word's few
  jpgs become one short pseudo-clip, label = that single word/phrase. This is
  isolated-word, not continuous-sentence data — weaker demo, but it's real trained
  weights on real data, proves the adapter pipeline end-to-end. Set
  `WORD_FRAMES_ROOT` to the `Frames_Word_Level` folder.

If `"video"` gives nothing and `"word_frames"` also looks wrong, check the original
Mendeley source (data.mendeley.com/datasets/kcmpdxky7p/1) — it may host the actual
sentence videos that this Kaggle mirror dropped.

**The next cell will refuse to run until you set these correctly.**

In [ ]:
# === FILL THIS IN — do not skip ===
DATA_MODE = None          # "video" or "word_frames"
VIDEO_ROOT = None         # e.g. f"{dataset_path}/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/Videos_Sentence_Level"
WORD_FRAMES_ROOT = None   # e.g. f"{dataset_path}/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/Frames_Word_Level"

assert DATA_MODE in ("video", "word_frames"), (
    "Set DATA_MODE above based on cell 4's output, then re-run this cell."
)
chosen_root = VIDEO_ROOT if DATA_MODE == "video" else WORD_FRAMES_ROOT
assert chosen_root is not None and os.path.isdir(chosen_root), (
    f"Set the matching *_ROOT to a real folder path (got: {chosen_root!r})."
)
print("confirmed mode:", DATA_MODE, "->", chosen_root)

In [ ]:
# 6a. VIDEO mode: build ISLTranslate.csv + flat <uid>.mp4 folder, then use the
# repo's extract_keypoints.py as-is. Skipped entirely if DATA_MODE != "video".
# Wipes any leftover copies from a previous run first — this is what was
# actually piling up on your working-storage quota, not the mount itself.
import glob, shutil
import pandas as pd

os.chdir(REPO_ROOT)
if DATA_MODE == "video":
    RAW_VIDEOS_DIR = "data/raw_videos"
    shutil.rmtree(RAW_VIDEOS_DIR, ignore_errors=True)
    os.makedirs(RAW_VIDEOS_DIR, exist_ok=True)

    video_files = []
    for ext in ("mp4", "MP4", "avi", "AVI", "mov", "MOV"):
        video_files += glob.glob(os.path.join(VIDEO_ROOT, "**", f"*.{ext}"), recursive=True)
    print(f"found {len(video_files)} video files under VIDEO_ROOT")

    # ADJUST if the sentence isn't the immediate parent folder name
    def sentence_id_from_path(video_path: str) -> str:
        return os.path.basename(os.path.dirname(video_path))

    rows = []
    for i, vp in enumerate(video_files):
        uid = f"clip{i:04d}"
        text = sentence_id_from_path(vp).replace("_", " ").strip()
        if not text:
            continue
        dst = os.path.join(RAW_VIDEOS_DIR, f"{uid}.mp4")
        if not os.path.exists(dst):
            shutil.copy(vp, dst)
        rows.append({"uid": uid, "text": text})

    df = pd.DataFrame(rows)
    os.makedirs("data/labels", exist_ok=True)
    df.to_csv("data/labels/ISLTranslate.csv", index=False)
    print(df.shape)
    display(df.head(10))
else:
    print("skipped — DATA_MODE is word_frames")

**If DATA_MODE == "video": check `df.head(10)` above — `text` must read like real
words/sentences, not garbage. If wrong, fix `sentence_id_from_path()` and re-run.**

In [ ]:
# 6b. VIDEO mode only: run the repo's own MediaPipe extraction script
if DATA_MODE == "video":
    !python backend/scripts/extract_keypoints.py \
      --videos_dir data/raw_videos \
      --labels_csv data/labels/ISLTranslate.csv \
      --out_dir data/processed/isltranslate
else:
    print("skipped — DATA_MODE is word_frames")

In [ ]:
# 6c. WORD_FRAMES mode only: each word folder's jpgs become one short pseudo-clip.
# No existing script handles this (extract_keypoints.py is video-only), so this
# runs MediaPipe on the still images directly and writes the exact same
# pose/<uid>.npy + face/<uid>.npy + ISLTranslate.csv layout the trainer expects.
import numpy as np

os.chdir(REPO_ROOT)
if DATA_MODE == "word_frames":
    import cv2
    import mediapipe as mp

    out_dir = "data/processed/isltranslate"
    pose_dir = os.path.join(out_dir, "pose")
    face_dir = os.path.join(out_dir, "face")
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(pose_dir, exist_ok=True)
    os.makedirs(face_dir, exist_ok=True)

    mp_holistic = mp.solutions.holistic
    word_folders = sorted(
        d for d in os.listdir(WORD_FRAMES_ROOT)
        if os.path.isdir(os.path.join(WORD_FRAMES_ROOT, d))
    )
    print(f"found {len(word_folders)} word folders")

    rows = []
    with mp_holistic.Holistic(static_image_mode=True, model_complexity=1) as holistic:
        for word in word_folders:
            word_dir = os.path.join(WORD_FRAMES_ROOT, word)
            imgs = sorted(
                f for f in os.listdir(word_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            )
            if not imgs:
                continue

            pose_frames, face_frames = [], []
            for img_name in imgs:
                img = cv2.imread(os.path.join(word_dir, img_name))
                if img is None:
                    continue
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                results = holistic.process(rgb)

                if results.pose_landmarks:
                    pose_frames.append(np.array(
                        [[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark],
                        dtype=np.float32).flatten())
                else:
                    pose_frames.append(np.zeros(33 * 4, dtype=np.float32))

                if results.face_landmarks:
                    face_frames.append(np.array(
                        [[lm.x, lm.y, lm.z] for lm in results.face_landmarks.landmark],
                        dtype=np.float32).flatten())
                else:
                    face_frames.append(np.zeros(478 * 3, dtype=np.float32))

            if not pose_frames:
                print(f"  skip {word}: no frames readable")
                continue

            uid = word.strip().lower().replace(" ", "_").replace("/", "_").replace("'", "")
            np.save(os.path.join(pose_dir, f"{uid}.npy"), np.stack(pose_frames))
            np.save(os.path.join(face_dir, f"{uid}.npy"), np.stack(face_frames))
            text = word.strip().lower().replace("_", " ")
            rows.append({"uid": uid, "text": text})
            print(f"[{len(rows)}] {word} -> {len(pose_frames)} frames")

    df = pd.DataFrame(rows)
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(os.path.join(out_dir, "ISLTranslate.csv"), index=False)
    print(df.shape)
    display(df.head(10))
else:
    print("skipped — DATA_MODE is video")

In [ ]:
# 7. Train — small model, small dataset, modest epochs for free-tier GPU time.
%cd {REPO_ROOT}
!PYTHONPATH=backend python -m app.training.train_base_model \
  --data-dir data/processed/isltranslate \
  --output backend/app/models/weights/base_model.pt \
  --epochs 15 \
  --batch-size 4

In [ ]:
# 8a. Download the two output files to your machine
OUT_PT = "backend/app/models/weights/base_model.pt"
OUT_VOCAB = "backend/app/models/weights/base_model.vocab.json"

try:
    from google.colab import files
    files.download(OUT_PT)
    files.download(OUT_VOCAB)
except ImportError:
    # Kaggle: copy into /kaggle/working (already the case if run from repo root
    # under /kaggle/working) — download both from the notebook's Output/Data
    # pane after a run, or right-click > Download in the file browser.
    print("On Kaggle: download these two files from the Output pane —")
    print(OUT_PT)
    print(OUT_VOCAB)
print("drop both into backend/app/models/weights/ in your local repo, then commit+push.")

## 8b. (optional) push straight from Colab instead
Only run this if you want to skip the manual download/upload step. It asks for a token at runtime (not stored in this notebook) — use a **fine-grained GitHub PAT scoped to just this repo**, and revoke it after.

In [ ]:
import getpass

token = getpass.getpass("GitHub token (input hidden, not saved to notebook): ")
!git config user.email "colab-trainer@users.noreply.github.com"
!git config user.name "SilentBridge Colab Trainer"
!git add -f backend/app/models/weights/base_model.pt backend/app/models/weights/base_model.vocab.json
!git commit -m "Train base_model.pt on ISL-CSLTR (real weights, replaces placeholder)"
!git push https://{token}@github.com/BharathWaj-K-R/SilentBridge.git main
del token